# Learned Stage-Wise Curvature Tuning on Beans

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Beans image-classification dataset using ImageNet-pretrained ResNet models.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — the original Curvature Tuning method, which searches for one global beta value using validation performance.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters, all initialized at beta = 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

The purpose of testing both fixed and random initialization is to determine whether the performance of Learned SW-CT depends on the previously selected beta = 0.78 starting point.

Each method is evaluated using seeds 42, 43, and 44.

Experiments are first conducted for 20 epochs and then repeated for 30 epochs to determine whether additional optimization improves accuracy or changes the learned stage-wise curvature pattern.

Wall-clock execution time is also recorded for every method to compare computational cost.

The experiments are repeated using:

- ResNet-18
- ResNet-50
- ResNet-152

In [1]:
!nvidia-smi

Wed Sep 30 22:54:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 6.49 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 65.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 M

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Beans Dataset

Before running the experiments, the Beans data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 1034
Validation samples: 133
Test samples: 128
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Beans dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 22:56:52.132 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed42_epochs20.log
2026-09-30 22:56:52.135 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 171MB/s]
2026-09-30 22:56:54.091 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 22:56:54.133 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-09-30 22:56:54.134 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 22:57:01.226 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.114670, Accuracy: 40.62%
2026-09-30 22:57:01.457 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.633725, Val Accuracy: 81.95%
2026-09-30 22:57:01.458 | INFO     | train:linear_probe:190 - New best validation accuracy: 81.95 a

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 23:04:43.220 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed43_epochs20.log
2026-09-30 23:04:43.222 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:04:44.410 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:04:44.452 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-09-30 23:04:44.452 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:04:50.861 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.262046, Accuracy: 28.12%
2026-09-30 23:04:51.077 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.660274, Val Accuracy: 80.45%
2026-09-30 23:04:51.078 | INFO     | train:linear_probe:190 - New best validation accuracy: 80.45 at epoch 1
2026-09-30 23:04:51.140 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.660985, Accuracy: 75.00%
2026-09-30 23:04:51.367 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.441943, Val Ac

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 23:12:47.480 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed44_epochs20.log
2026-09-30 23:12:47.482 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:12:48.677 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:12:48.718 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-09-30 23:12:48.718 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:12:55.543 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.176766, Accuracy: 25.00%
2026-09-30 23:12:55.786 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.671520, Val Accuracy: 77.44%
2026-09-30 23:12:55.787 | INFO     | train:linear_probe:190 - New best validation accuracy: 77.44 at epoch 1
2026-09-30 23:12:55.857 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.730835, Accuracy: 75.00%
2026-09-30 23:12:56.095 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.457545, Val Ac

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:20:42.617 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:20:42.670 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:20:43.723 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:20:43.723 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:20:43.771 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:20:43.772 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:20:43.772 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:20:45.334 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.085081, Accuracy: 34.38%
2026-09-30 23:21:04.889 | INFO     | train:test_epoch:90 -

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:27:40.136 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:27:40.188 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:27:41.244 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:27:41.245 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:27:41.297 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:27:41.297 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:27:41.298 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:27:42.679 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.162185, Accuracy: 28.12%
2026-09-30 23:28:02.097 | INFO     | train:test_epoch:90 -

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:34:38.485 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:34:38.544 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:34:39.596 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:34:39.596 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:34:39.649 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:34:39.649 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:34:39.650 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:34:41.075 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.076154, Accuracy: 37.50%
2026-09-30 23:35:00.594 | INFO     | train:test_epoch:90 -

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:41:36.653 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-09-30 23:41:36.705 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:41:37.784 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:41:37.784 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:41:37.834 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-09-30 23:41:37.834 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:41:37.835 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:41:39.187 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.074683, Accuracy: 40.62%
2026-09-30 23:41:58.658 | INFO     | train:tes

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:48:34.357 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-09-30 23:48:34.409 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:48:35.444 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:48:35.444 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:48:35.499 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-09-30 23:48:35.500 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:48:35.500 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:48:36.904 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.158476, Accuracy: 25.00%
2026-09-30 23:48:56.380 | INFO     | train:t

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:55:35.907 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-09-30 23:55:35.963 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:55:37.192 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:55:37.192 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-09-30 23:55:37.291 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-09-30 23:55:37.292 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:55:37.292 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-09-30 23:55:38.930 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.075994, Accuracy: 40.62%
2026-09-30 23:55:58.779 | INFO     | train:t

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-10-01 00:02:41.089 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed42_epochs30.log
2026-10-01 00:02:41.091 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 00:02:42.366 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 00:02:42.413 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-10-01 00:02:42.413 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 00:02:49.520 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.114670, Accuracy: 40.62%
2026-10-01 00:02:49.753 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.633725, Val Accuracy: 81.95%
2026-10-01 00:02:49.754 | INFO     | train:linear_probe:190 - New best validation accuracy: 81.95 at epoch 1
2026-10-01 00:02:49.816 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.624192, Accuracy: 84.38%
2026-10-01 00:02:50.077 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.427351, Val Ac

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-10-01 00:13:32.140 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed43_epochs30.log
2026-10-01 00:13:32.143 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 00:13:33.470 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 00:13:33.511 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-10-01 00:13:33.511 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 00:13:41.286 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.262046, Accuracy: 28.12%
2026-10-01 00:13:41.544 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.660274, Val Accuracy: 80.45%
2026-10-01 00:13:41.545 | INFO     | train:linear_probe:190 - New best validation accuracy: 80.45 at epoch 1
2026-10-01 00:13:41.624 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.660985, Accuracy: 75.00%
2026-10-01 00:13:41.897 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.441943, Val Ac

In [14]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-10-01 00:24:17.168 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet50_seed44_epochs30.log
2026-10-01 00:24:17.171 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 00:24:18.514 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 00:24:18.554 | INFO     | __main__:main:116 - Number of trainable parameters: 6147
2026-10-01 00:24:18.554 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 00:24:25.897 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.176766, Accuracy: 25.00%
2026-10-01 00:24:26.184 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.671520, Val Accuracy: 77.44%
2026-10-01 00:24:26.185 | INFO     | train:linear_probe:190 - New best validation accuracy: 77.44 at epoch 1
2026-10-01 00:24:26.257 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.730835, Accuracy: 75.00%
2026-10-01 00:24:26.551 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.457545, Val Ac

In [15]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:35:37.493 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:35:37.547 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:35:38.618 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:35:38.619 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 00:35:38.673 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:35:38.674 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:35:38.674 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 00:35:40.295 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.085081, Accuracy: 34.38%
2026-10-01 00:36:00.302 | INFO     | train:test_epoch:90 -

In [16]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:46:03.532 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:46:03.584 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:46:04.685 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:46:04.686 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 00:46:04.736 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:46:04.737 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:46:04.737 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 00:46:06.081 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.162185, Accuracy: 28.12%
2026-10-01 00:46:25.586 | INFO     | train:test_epoch:90 -

In [17]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:56:45.351 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:56:45.405 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:56:46.499 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:56:46.499 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 00:56:46.551 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:56:46.552 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:56:46.552 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 00:56:47.908 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.076154, Accuracy: 37.50%
2026-10-01 00:57:07.470 | INFO     | train:test_epoch:90 -

In [18]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 01:07:24.443 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 01:07:24.552 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 01:07:26.114 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 01:07:26.115 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 01:07:26.199 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 01:07:26.200 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 01:07:26.200 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 01:07:28.144 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.074683, Accuracy: 40.62%
2026-10-01 01:07:48.159 | INFO     | train:tes

In [19]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 01:18:12.904 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-01 01:18:13.013 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 01:18:14.297 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 01:18:14.297 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 01:18:14.371 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 01:18:14.371 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 01:18:14.372 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 01:18:16.093 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.158476, Accuracy: 25.00%
2026-10-01 01:18:36.022 | INFO     | train:t

In [20]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 01:28:50.700 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-01 01:28:50.800 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 01:28:52.232 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 01:28:52.232 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-01 01:28:52.284 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 01:28:52.285 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 01:28:52.285 | INFO     | __main__:main:439 - Total trainable parameters: 6151
2026-10-01 01:28:53.793 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.075994, Accuracy: 40.62%
2026-10-01 01:29:13.422 | INFO     | train:t